# Deep Interpreter Literature Benchmark Smoke Notebook

This notebook compares `dynamic`, `baseline_lit`, and `baseline_eutils` on a small smoke subset (default: 3 cases).

- Keeps current methodology unchanged
- Changes only evaluation scope (small, fast run)
- Writes raw smoke records to `tests/res/` for manual inspection


In [ ]:
import os
import json
import time
from pathlib import Path
from datetime import datetime, UTC

from dotenv import load_dotenv

load_dotenv()
os.environ['LANGCHAIN_TRACING_V2'] = 'false'

print('LLM_PROVIDER=', os.getenv('LLM_PROVIDER', 'UNSET'))
print('OPENROUTER_API_KEY set=', bool(os.getenv('OPENROUTER_API_KEY')))
print('OPENAI_API_KEY set=', bool(os.getenv('OPENAI_API_KEY')))
print('BIOMCP_URL=', os.getenv('BIOMCP_URL', 'http://localhost:8000/mcp'))


In [ ]:
from scripts.benchmarks.run_lit_benchmark import load_cases, _run_method_once
from src.agents.deep_interpreter.tools.literature import _calculate_coverage

CASES_PATH = Path('docs/benchmarks/lit_repro_benchmark/cases_v1.json')
all_cases = load_cases(str(CASES_PATH))

SMOKE_CASE_COUNT = 3
smoke_cases = all_cases[:SMOKE_CASE_COUNT]

print('Loaded cases:', len(all_cases))
print('Smoke cases:', [c['case_id'] for c in smoke_cases])


In [ ]:
METHODS = ('dynamic', 'baseline_lit', 'baseline_eutils')
DYNAMIC_REPEATS = 2
BASELINE_REPEATS = 1
MAX_RESULTS = 30
MAX_ITERATIONS = 3
RERANK_MODE = 'off'

def coverage_pass(coverage):
    if not isinstance(coverage, dict):
        return False
    gene_cov = float(coverage.get('genes_coverage_pct', 0.0) or 0.0)
    path_cov = float(coverage.get('pathways_coverage_pct', 0.0) or 0.0)
    total_papers = int(coverage.get('total_papers', 0) or 0)
    return gene_cov >= 60.0 and path_cov >= 50.0 and total_papers >= 15

def run_smoke(cases):
    records = []
    total = sum((DYNAMIC_REPEATS if m == 'dynamic' else BASELINE_REPEATS) for _ in cases for m in METHODS)
    done = 0

    for case in cases:
        for method in METHODS:
            repeats = DYNAMIC_REPEATS if method == 'dynamic' else BASELINE_REPEATS
            for rep in range(repeats):
                done += 1
                started = time.perf_counter()
                err = None

                try:
                    papers, coverage, query_used, search_method, recommended_actions = _run_method_once(
                        method=method,
                        case=case,
                        max_results=MAX_RESULTS,
                        max_iterations=MAX_ITERATIONS,
                        rerank_mode=RERANK_MODE,
                    )
                except Exception as e:
                    papers = []
                    coverage = {}
                    query_used = None
                    search_method = None
                    recommended_actions = []
                    err = str(e)

                latency_ms = (time.perf_counter() - started) * 1000.0

                rec = {
                    'case_id': case['case_id'],
                    'method': method,
                    'repeat_index': rep,
                    'total_papers': len(papers),
                    'coverage': coverage,
                    'coverage_pass': coverage_pass(coverage),
                    'latency_ms': latency_ms,
                    'search_method': search_method,
                    'query_used': query_used,
                    'recommended_actions': recommended_actions,
                    'error': err,
                    'papers': papers,
                }
                records.append(rec)

                print(f"[{done}/{total}] {method} case={case['case_id']} rep={rep} papers={len(papers)} error={'yes' if err else 'no'}")

    return records

records = run_smoke(smoke_cases)
print('Done. records=', len(records))


In [ ]:
from collections import defaultdict

def mean(vals):
    return (sum(vals) / len(vals)) if vals else 0.0

summary = defaultdict(list)
for r in records:
    summary[r['method']].append(r)

rows = []
for method, rs in sorted(summary.items()):
    gene_cov = [float((r.get('coverage') or {}).get('genes_coverage_pct', 0.0) or 0.0) for r in rs]
    path_cov = [float((r.get('coverage') or {}).get('pathways_coverage_pct', 0.0) or 0.0) for r in rs]
    papers = [int(r.get('total_papers', 0) or 0) for r in rs]
    pass_vals = [1.0 if r.get('coverage_pass') else 0.0 for r in rs]

    rows.append({
        'method': method,
        'n_runs': len(rs),
        'avg_total_papers': round(mean(papers), 3),
        'avg_gene_cov_pct': round(mean(gene_cov), 3),
        'avg_path_cov_pct': round(mean(path_cov), 3),
        'coverage_pass_rate': round(mean(pass_vals), 3),
        'avg_latency_ms': round(mean([float(r.get('latency_ms', 0.0) or 0.0) for r in rs]), 3),
    })

rows


In [ ]:
# Top-K focused comparison: evaluate first K returned papers (default: 5/10/15)
TOP_KS = (5, 10, 15)

topk_rows = []
for k in TOP_KS:
    for method, rs in sorted(summary.items()):
        gvals = []
        pvals = []
        both = []
        for r in rs:
            inp = next(c for c in smoke_cases if c['case_id'] == r['case_id'])
            topk = (r.get('papers') or [])[:k]
            covk = _calculate_coverage(
                papers=topk,
                genes=inp.get('genes', []),
                pathways=inp.get('pathways', []),
                celltypes=inp.get('celltypes', []),
            )
            g = float(covk.get('genes_coverage_pct', 0.0) or 0.0)
            p = float(covk.get('pathways_coverage_pct', 0.0) or 0.0)
            gvals.append(g)
            pvals.append(p)
            both.append(1.0 if (g >= 60.0 and p >= 50.0) else 0.0)

        topk_rows.append({
            'top_k': k,
            'method': method,
            f'gene_cov@{k}_mean': round(mean(gvals), 3),
            f'path_cov@{k}_mean': round(mean(pvals), 3),
            f'both_hit@{k}_rate': round(mean(both), 3),
        })

# Keep backward-compatible top10 slice for convenience
top10_rows = [r for r in topk_rows if r['top_k'] == 10]

# Full Top-K comparison table
topk_rows



In [ ]:
# Save records for manual inspection
out_dir = Path('tests/res')
out_dir.mkdir(parents=True, exist_ok=True)
ts = datetime.now(UTC).strftime('%Y%m%dT%H%M%SZ')
jsonl_path = out_dir / f'literature_smoke_compare_{ts}.jsonl'
summary_path = out_dir / f'literature_smoke_compare_{ts}_summary.json'

with jsonl_path.open('w', encoding='utf-8') as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

payload = {
    'generated_at_utc': datetime.now(UTC).isoformat(),
    'smoke_case_ids': [c['case_id'] for c in smoke_cases],
    'rows': rows,
    'topk_rows': topk_rows,
    'top10_rows': top10_rows,
}
summary_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')

print('Saved:', jsonl_path)
print('Saved:', summary_path)



## Notes

- This notebook defaults to smoke mode (3 cases).
- Top-K comparison defaults to `TOP_KS = (5, 10, 15)`.
- To run larger checks, increase `SMOKE_CASE_COUNT` and repeats.
- For full benchmark runs, use `python -m scripts.benchmarks.run_lit_benchmark` and `score_lit_benchmark`.

